<a href="https://colab.research.google.com/github/DbYet/Zero-Shot-Classification-Data-and-Code-/blob/main/formatting_%26_Zero_Shot_Classification_Code_for_HAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PHASE 1: ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
# Run this cell in Google Colab to install the Hugging Face ecosystem
!pip install -q transformers datasets accelerate tqdm pandas sklearn

import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from google.colab import files # Imported for auto-downloading

# ==============================================================================
# ZIP EXTRACTION AUTOMATION
# ==============================================================================
if os.path.exists("archive.zip"):
    print("[0/4] Unzipping archive.zip file...")
    !unzip -o archive.zip
else:
    print("[0/4] No archive.zip found, looking directly for csv files.")

# ==============================================================================
# DYNAMIC FILE SPECIFICATION
# ==============================================================================
INPUT_FILE_NAME = "spam.csv"

base_name = os.path.splitext(INPUT_FILE_NAME)[0]
OUTPUT_PREDICTIONS_FILE = f"{base_name}_predictions.csv"
OUTPUT_SUMMARY_FILE = f"{base_name}_ratio_summary.txt"

# Behavioral Guardrails
CHARACTER_LIMIT = 1500
BATCH_SIZE = 16

# Set up device-agnostic execution (GPU acceleration)
device = 0 if torch.cuda.is_available() else -1
print(f"Execution Device Detected: {'GPU (cuda:0)' if device == 0 else 'CPU'}")

# Define evaluation labels and models
candidate_labels = ["ham", "spam"]
models_to_evaluate = {
    "BART": "facebook/bart-large-mnli",
    "RoBERTa": "FacebookAI/roberta-large-mnli",
    "DeBERTa": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
}

# ==============================================================================
# DATA LOADING AND DYNAMIC STRATIFIED SAMPLING (OPTION B)
# ==============================================================================
print("\n[1/4] Loading, cleaning, and sampling text inputs...")
if not os.path.exists(INPUT_FILE_NAME):
    raise FileNotFoundError(f"Could not find '{INPUT_FILE_NAME}'. Please ensure archive.zip contained spam.csv.")

df = pd.read_csv(INPUT_FILE_NAME, encoding='latin-1')

# Restructure columns and drop hidden empty 'Unnamed' columns
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    df = df.rename(columns={'v1': 'Spam/Ham', 'v2': 'Message'})

if 'Message' not in df.columns:
    raise KeyError("Could not find an appropriate text column named 'Message' in the dataset.")

# Clean missing entries safely
df['Message'] = df['Message'].fillna("")
df['Processed_Text'] = df['Message'].astype(str).str.slice(0, CHARACTER_LIMIT)
df = df[df['Processed_Text'].str.strip() != ""].reset_index(drop=True)

# --- OPTION B: STRATIFIED SAMPLING (1,000 Messages Total) ---
# This preserves the exact real-world ratio of Ham vs Spam while ensuring swift runtime.
total_sample_size = 1000
ham_fraction = df['Spam/Ham'].value_counts(normalize=True).get('ham', 0.866)
ham_count = int(total_sample_size * ham_fraction)
spam_count = total_sample_size - ham_count

df_ham = df[df['Spam/Ham'] == 'ham'].sample(n=ham_count, random_state=42)
df_spam = df[df['Spam/Ham'] == 'spam'].sample(n=spam_count, random_state=42)
df = pd.concat([df_ham, df_spam]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Stratified sampling finalized! Resized to {len(df)} rows for rapid execution.")
print(f" -> Sub-sample breakdown: {df['Spam/Ham'].value_counts().to_dict()}")

# ==============================================================================
# DECOUPLED INFERENCE PIPELINE
# ==============================================================================
results_dict = {
    "Message_Text": df['Processed_Text'].tolist(),
    "Ground_Truth": df['Spam/Ham'].tolist()
}

for model_name, model_repo in models_to_evaluate.items():
    print(f"\n[2/4] Initializing zero-shot pipeline for {model_name}...")
    try:
        classifier = pipeline(
            "zero-shot-classification",
            model=model_repo,
            device=device,
            model_kwargs={"torch_dtype": torch.float16} if device == 0 else {}
        )

        opinions = []
        certainties = []
        texts = df['Processed_Text'].tolist()

        print(f"[3/4] Running zero-shot inference for {model_name}...")
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"{model_name}"):
            batch_texts = texts[i:i + BATCH_SIZE]

            outputs = classifier(
                batch_texts,
                candidate_labels=candidate_labels,
                hypothesis_template="This text is {}."
            )

            if not isinstance(outputs, list):
                outputs = [outputs]

            for output in outputs:
                opinions.append(output['labels'][0])
                certainties.append(round(output['scores'][0], 4))

        results_dict[f"{model_name}_Opinion"] = opinions
        results_dict[f"{model_name}_Certainty"] = certainties

        del classifier
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"CRITICAL ERROR executing {model_name}: {e}")
        results_dict[f"{model_name}_Opinion"] = [np.nan] * len(df)
        results_dict[f"{model_name}_Certainty"] = [np.nan] * len(df)

# ==============================================================================
# FILE EXPORT AND RATIO GENERATION
# ==============================================================================
print("\n[4/4] Writing guesses and calculating macro ratios...")
df_output = pd.DataFrame(results_dict)

summary_lines = [f"Summary Metrics for Resampled Input Dataset: {INPUT_FILE_NAME}\n" + "="*50 + "\n"]

for model_name in models_to_evaluate.keys():
    op_col = f"{model_name}_Opinion"
    counts = df_output[op_col].value_counts()

    spam_count = counts.get('spam', 0)
    ham_count = counts.get('ham', 0)
    ratio = spam_count / ham_count if ham_count > 0 else float('inf')

    metric_string = (
        f"Model: {model_name}\n"
        f"  -> Predicted SPAM: {spam_count}\n"
        f"  -> Predicted HAM:  {ham_count}\n"
        f"  -> Predicted Spam-to-Ham Ratio: {ratio:.4f}\n"
    )
    print(metric_string)
    summary_lines.append(metric_string)

df_output.to_csv(OUTPUT_PREDICTIONS_FILE, index=False)
with open(OUTPUT_SUMMARY_FILE, "w") as f:
    f.writelines(summary_lines)

print(f"\nInference run complete in record time!")
print(f"-> Resized professional guesses saved to: {OUTPUT_PREDICTIONS_FILE}")

# ==============================================================================
# AUTOMATIC FILE DOWNLOAD
# ==============================================================================
print("\nInitiating automatic file downloads...")
try:
    files.download(OUTPUT_PREDICTIONS_FILE)
    files.download(OUTPUT_SUMMARY_FILE)
    print("Downloads triggered successfully!")
except Exception as e:
    print(f"Warning: Automatic download failed. You may need to download files manually from the left panel. Error: {e}")

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
[0/4] Unzipping archive.zip file...
Archive:  archive.zip
  inflating: spam.csv                
Execution Device Detected: GPU (cuda:0)

[1/4] Loading, cleaning, and sampling text inputs...
Stratified sampling finalized! Resized to 1000 rows for rapid execution.
 -> Sub-sample breakdown: {'ham': 865, 'spam': 135}

[2/4] Initializing zero-shot pipeline for BART...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads triggered successfully!
